## Neural Machine Translation (NMT)
This notebook demonstrates the sequence-to-sequence model with the case study of neural machine translation. The model is an LSTM based encoder-decoder model with attention. \\
Here [OpenNMT-Py](https://github.com/OpenNMT/OpenNMT-py) toolkit is used for training the NMT model.

### Setting up

When running this for the first time you may get a warning telling you to restart the Runtime. You can ignore this, but feel free to select "Runtime->Restart Runtime" from the overhead menu if you encounter problems.

In [1]:
# install openNMT-Py from github
# !pip3 install -q git+https://github.com/OpenNMT/OpenNMT-py.git
!pip install OpenNMT-py
# sacrebleu for evaluation
!pip install -q install sacrebleu

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.8/262.8 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.6/38.6 MB 24.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.0/17.0 MB 26.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 755.6/755.6 MB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 117.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 86.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 59.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 731.7/731.7 MB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [14]:
## Download the dataset
!wget https://s3.amazonaws.com/opennmt-trainingdata/toy-ende.tar.gz
!tar xf toy-ende.tar.gz

--2025-06-08 11:47:19--  https://s3.amazonaws.com/opennmt-trainingdata/toy-ende.tar.gz
Resolving s3.amazonaws.com (s3.amazonaws.com)... 54.231.135.248, 54.231.165.16, 52.217.199.208, ...
Connecting to s3.amazonaws.com (s3.amazonaws.com)|54.231.135.248|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1662081 (1.6M) [application/x-gzip]
Saving to: ‘toy-ende.tar.gz.1’

toy-ende.tar.gz.1   100%[===================>]   1.58M  4.43MB/s    in 0.4s    

2025-06-08 11:47:20 (4.43 MB/s) - ‘toy-ende.tar.gz.1’ saved [1662081/1662081]



## Data instances

In [15]:
!cd toy-ende/ && ls

pred_50.txt  src-test.txt   src-val.txt   tgt-train.txt
run	     src-train.txt  tgt-test.txt  tgt-val.txt


In [16]:
# number of instances
!echo "Number of lines:" && wc -l toy-ende/src-train.txt

Number of lines:
10000 toy-ende/src-train.txt


## Configuration


In [17]:
# Create the YAML configuration file
# On a regular machine, you can create it manually or with nano

config = '''# toy_en_de.yaml

## Where the samples will be written
save_data: toy-ende/run/example

## Where the vocab(s) will be written
src_vocab: toy-ende/run/example.vocab.src
tgt_vocab: toy-ende/run/example.vocab.tgt

## Where the model will be saved
save_model: model/model

# Prevent overwriting existing files in the folder
overwrite: False

# Corpus opts:
data:
    corpus_1:
        path_src: toy-ende/src-train.txt
        path_tgt: toy-ende/tgt-train.txt
    valid:
        path_src: toy-ende/src-val.txt
        path_tgt: toy-ende/tgt-val.txt

world_size: 1
gpu_ranks: [0]

# Remove or modify these lines for bigger files
train_steps: 50
valid_steps: 25
'''

with open("toy_en_de.yaml", "w+") as config_yaml:
  config_yaml.write(config)

!cat toy_en_de.yaml

# toy_en_de.yaml

## Where the samples will be written
save_data: toy-ende/run/example

## Where the vocab(s) will be written
src_vocab: toy-ende/run/example.vocab.src
tgt_vocab: toy-ende/run/example.vocab.tgt

## Where the model will be saved
save_model: model/model

# Prevent overwriting existing files in the folder
overwrite: False

# Corpus opts:
data:
    corpus_1:
        path_src: toy-ende/src-train.txt
        path_tgt: toy-ende/tgt-train.txt
    valid:
        path_src: toy-ende/src-val.txt
        path_tgt: toy-ende/tgt-val.txt

world_size: 1
gpu_ranks: [0]

# Remove or modify these lines for bigger files
train_steps: 50
valid_steps: 25


## Vocabulary buildup

In [18]:
# Build Vocabulary

!onmt_build_vocab -config toy_en_de.yaml -n_sample -1


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.0.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/usr/local/bin/onmt_build_vocab", line 5, in <module>
    from onmt.bin.build_vocab import main
  File "/usr/local/lib/python3.11/dist-packages/onmt/__init__.py", line 2, in <module>
    import onmt.inputters
  File "/usr/local/lib/python3.11/dist-packages/onmt/inputters/__init__.py", line 7, in <module>
    from onmt.inputters.text_utils import text_sort_key, process, numericalize, tensorify
  File "/usr/local/lib/python3.11/dist-packages/onmt/inputters/text_utils.py", line 1, in <module>
    import torch
  File "

## Training

In [19]:
!onmt_train -config toy_en_de.yaml


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.0.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/usr/local/bin/onmt_train", line 5, in <module>
    from onmt.bin.train import main
  File "/usr/local/lib/python3.11/dist-packages/onmt/__init__.py", line 2, in <module>
    import onmt.inputters
  File "/usr/local/lib/python3.11/dist-packages/onmt/inputters/__init__.py", line 7, in <module>
    from onmt.inputters.text_utils import text_sort_key, process, numericalize, tensorify
  File "/usr/local/lib/python3.11/dist-packages/onmt/inputters/text_utils.py", line 1, in <module>
    import torch
  File "/usr/local/l

## Inferencing

In [8]:
!onmt_translate -model model/model_step_50.pt -src toy-ende/src-test.txt -output toy-ende/pred_50.txt -gpu 0 -verbose

Streaming output truncated to the last 5000 lines.
PRED 1738: die die in die die , die die die die die die die die die
PRED SCORE: -0.9130

[2025-06-08 11:05:28,976 INFO] 
SENT 1739: ['Pope', 'Francis', 'will', 'create', 'new', '<unk>', 'of', 'the', 'Catholic', 'Church', 'for', 'his', 'first', 'time', 'on', 'February', '22', ',', 'the', '<unk>', 'announced', 'Thursday', '.']
PRED 1739: die die in die die , die die die die die die die die die die die die die die die die die die die die die die die die die die die
PRED SCORE: -0.8364

[2025-06-08 11:05:28,977 INFO] 
SENT 1740: ['<unk>', 'are', 'the', '<unk>', 'clergy', 'in', 'the', 'Catholic', 'Church', 'below', 'the', '<unk>', ',', 'and', 'they', '&apos;re', 'the', 'ones', 'who', '<unk>', '<unk>', ',', 'so', 'Francis', 'will', 'be', 'appointing', 'his', 'first', 'group', 'of', 'men', 'who', 'will', 'ultimately', 'help', 'choose', 'his', 'successor', '.']
PRED 1740: die die in die die , die die die die die die die die die die die die die

Evaluation with BLEU using [SacreBLEU](https://github.com/mjpost/sacrebleu)

In [10]:
!sacrebleu toy-ende/tgt-test.txt < toy-ende/pred_50.txt

sacreBLEU: That's 100 lines that end in a tokenized period ('.')
sacreBLEU: It looks like you forgot to detokenize your test data, which may hurt your score.
sacreBLEU: If you insist your data is detokenized, or don't care, you can suppress this message with the `force` parameter.
{
 "name": "BLEU",
 "score": 0.0,
 "signature": "nrefs:1|case:mixed|eff:no|tok:13a|smooth:exp|version:2.5.1",
 "verbose_score": "11.5/0.0/0.0/0.0 (BP = 0.400 ratio = 0.522 hyp_len = 30447 ref_len = 58319)",
 "nrefs": "1",
 "case": "mixed",
 "eff": "no",
 "tok": "13a",
 "smooth": "exp",
 "version": "2.5.1"
}


In [11]:
!sacrebleu toy-ende/tgt-test.txt < toy-ende/tgt-test.txt

sacreBLEU: That's 100 lines that end in a tokenized period ('.')
sacreBLEU: It looks like you forgot to detokenize your test data, which may hurt your score.
sacreBLEU: If you insist your data is detokenized, or don't care, you can suppress this message with the `force` parameter.
{
 "name": "BLEU",
 "score": 100.0,
 "signature": "nrefs:1|case:mixed|eff:no|tok:13a|smooth:exp|version:2.5.1",
 "verbose_score": "100.0/100.0/100.0/100.0 (BP = 1.000 ratio = 1.000 hyp_len = 58319 ref_len = 58319)",
 "nrefs": "1",
 "case": "mixed",
 "eff": "no",
 "tok": "13a",
 "smooth": "exp",
 "version": "2.5.1"
}


## Optional tasks


1.   Modify and tweak with the hyperameters (hidden_dim, number of  encoder-decoder layers, number of steps etc), model type (LSTM, Bi-LSTM, Transformers etc) in the config.yaml. Following are some pointers/references to follow up

    a.  [Training a transformer based NMT](https://opennmt.net/OpenNMT-py/FAQ.html#how-do-i-train-the-transformer-model)

    b.  [Documentation to the available hyperparameter configs](https://opennmt.net/OpenNMT-py/options/train.html)

2.   Using other dataset for Indic languages

    a.  [Samananter Indic-Indic](https://ai4b-my.sharepoint.com/:u:/g/personal/sumanthdoddapaneni_ai4bharat_org/Ef_zku9uvP5CpXXXcpGGVKEBxc4JOVlda2rVJxDIaOcySw?e=w3FW6J&download=1)
    
    b.  [Pmindia dataset](https://data.statmt.org/pmindia/v1/parallel/)

